# Lecția 05 - RAG agențial


## Configurare

Acest caiet demonstrează modelul Agentic RAG (Generare Augmentată prin Recuperare) folosind Microsoft Agent Framework.

**Precondiții:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — punctul de acces al serviciului dvs. Azure AI Search
- `AZURE_SEARCH_API_KEY` — cheia API pentru Azure AI Search
- Implementarea Azure OpenAI configurată prin variabile de mediu
- Azure CLI autentificat (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Ce este Agentic RAG?

RAG tradițional urmează un flux fix: preia documente, apoi generează un răspuns. **Agentic RAG** merge mai departe oferind agentului autonomie pentru a decide **când** și **cum** să preia informații.

Cu Agentic RAG, agentul poate:
- **Decide** dacă este nevoie de preluare înainte de a răspunde la o întrebare
- **Alege** sursa de date sau instrumentul de interogat
- **Evaluează** rezultatele preluate și realizează preluări suplimentare dacă prima tentativă este insuficientă
- **Combină** informațiile din mai mulți pași de preluare într-un răspuns coerent

Acest lucru face agentul mai flexibil și mai precis în comparație cu un flux static de preluare-urmarea generării.


## Crearea unui Instrument de Căutare

În Agentic RAG, sursele externe de date sunt încapsulate ca **instrumente** pe care agentul le poate invoca la cerere. Acest lucru permite agentului să trateze recuperarea ca pe o acțiune obișnuită pe care o poate întreprinde, și nu ca pe un pas obligatoriu.

Mai jos definim o bază de cunoștințe despre călătorii și o expunem ca un instrument pe care agentul îl poate apela pentru a căuta informații despre destinații.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Construirea Agentului RAG

Acum creăm un agent care este instruit să **obțină întotdeauna informații înainte de a răspunde**. Agentul folosește instrumentul `search_travel_knowledge` pentru a-și fundamenta răspunsurile în baza de cunoștințe, în loc să se bazeze pe propriile date de antrenament.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Căutare iterativă — Modelul Maker-Checker

Un avantaj cheie al Agentic RAG este **căutarea iterativă**. Agentul poate efectua mai multe runde de căutare pentru a verifica, rafina sau extinde constatările inițiale — similar unui flux de lucru de tip "maker-checker":

1. **Pasul maker**: Agentul extrage informații inițiale și redactează un răspuns.
2. **Pasul checker**: Agentul efectuează căutări suplimentare pentru a verifica detalii sau a completa goluri.

Mai jos, agentului i se pune o întrebare care necesită compararea mai multor destinații, determinându-l să caute de mai multe ori.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Rezumat

În această lecție ați învățat cum să construiți un sistem **Agentic RAG** folosind Microsoft Agent Framework:

- **Agentic RAG** permite agenților să decidă în mod autonom când să recupereze informații, făcând recuperarea dinamică, nu fixă.
- **Instrumente ca surse de date**: Baze externe de cunoștințe (precum Azure AI Search) sunt integrate ca instrumente pe care agentul le poate invoca.
- **Recuperare iterativă**: Modelul maker-checker permite agentului să efectueze mai multe runde de recuperare — căutare, verificare și rafinare — înainte de a produce un răspuns final.

În producție, ați înlocui baza de date în memorie `TRAVEL_KNOWLEDGE_BASE` cu un index real Azure AI Search pentru a gestiona recuperarea documentelor de călătorie la scară largă.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinarea răspunderii**:  
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). Deși ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autoritară. Pentru informații critice, se recomandă o traducere profesională realizată de un traducător uman. Nu ne asumăm răspunderea pentru eventualele neînțelegeri sau interpretări eronate rezultate din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
